Thyrosim Demo Notebook

For testing and running main Thyrosim model

In [12]:
import numpy as np
import pandas as pd
from itertools import product
import matplotlib.pyplot as plt
from thyrosim_model import simulate_patient
from concurrent.futures import ProcessPoolExecutor
from tqdm import tqdm

In [ ]:
# Basic implementation
def compute_means(df, window=50):
    tail = df.tail(window)

    return {
        "FT4_mean": tail["FT4"].mean(),
        "FT3_mean": tail["FT3"].mean(),
        "TT3_mean": tail["TT3"].mean(),
        "TSH_mean": tail["TSH"].mean(),
    }


def generate_full_dataset():
    # Full sweep
    #heights = list(range(150, 185, 5))
    #weights = list(range(50, 75, 5))
    #sexes = ["male", "female"]
    #lt4_vals = list(range(25, 50))
    #lt3_vals = list(range(5, 10))
    #rtf_vals = np.linspace(0.0, 1.0, 10)

    # test sweep
    heights = [150, 180]
    weights = [50, 70]
    sexes = ["male", "female"]
    lt4_vals = [25, 50]
    lt3_vals = [5, 10]
    rtf_vals = np.linspace(0.0, 1.0, 2)

    # total number of runs
    total = (
        len(heights)
        * len(weights)
        * len(sexes)
        * len(lt4_vals)
        * len(lt3_vals)
        * len(rtf_vals)
    )

    print(f"Total simulations: {total}")

    rows = []
    count = 0

    for h, w, s, lt4, lt3, rtf in product(
        heights, weights, sexes, lt4_vals, lt3_vals, rtf_vals
    ):
        count += 1

        # percentage progress
        progress = (count / total) * 100

        if count % 50 == 0 or count == 1:
            print(f"[{count}/{total}] ({progress:.2f}%)")

        df = simulate_patient(
            height=h,
            weight=w,
            sex=s,
            lt4_dose=lt4,
            lt3_dose=lt3,
            rtf=rtf
        )

        means = compute_means(df)

        row = {
            "height": h,
            "weight": w,
            "sex": s,
            "lt4": lt4,
            "lt3": lt3,
            "RTF": rtf,
            "timeseries": df,
            **means
        }

        rows.append(row)

    dataset = pd.DataFrame(rows)

    return dataset


if __name__ == "__main__":
    dataset = generate_full_dataset()
    dataset.to_pickle("thyrosim_full_dataset.pkl")

    print("\nSaved dataset to thyrosim_full_dataset.pkl")

Total simulations: 64
[1/64] (1.56%)
[50/64] (78.12%)

Saved dataset to thyrosim_full_dataset.pkl


In [15]:
import os
os.chdir("..")

df = pd.read_pickle("thyrosim_full_dataset.pkl")

In [16]:
df

,height,weight,sex,lt4,lt3,RTF,timeseries,FT4_mean,FT3_mean,TT3_mean,TSH_mean
0,150,50,male,25,5,0.000000,time FT4 FT3 T...,45.131306,15.779671,0.790871,0.343070
1,150,50,male,25,5,0.111111,time FT4 FT3 T...,45.212436,15.772888,0.789534,0.341792
2,150,50,male,25,5,0.222222,time FT4 FT3 T...,45.329850,15.778770,0.788389,0.340556
3,150,50,male,25,5,0.333333,time FT4 FT3 T...,45.417145,15.774169,0.787094,0.339364
4,150,50,male,25,5,0.444444,time FT4 FT3 T...,45.512923,15.772608,0.785851,0.343933
...,...,...,...,...,...,...,...,...,...,...,...
87495,180,70,female,49,9,0.555556,time FT4 FT3 T...,93.634819,24.053006,0.718338,0.401721
87496,180,70,female,49,9,0.666667,time FT4 FT3 T...,93.802556,24.063828,0.717742,0.399657
87497,180,70,female,49,9,0.777778,time FT4 FT3 T...,93.929920,24.064343,0.717062,0.400726
87498,180,70,female,49,9,0.888889,time FT4 FT3 T...,94.061489,24.065932,0.716391,0.401807
